# Auditoria temporal — Bronze, Silver e Gold

Consultas somente leitura para conferir competências disponíveis, identificar lacunas e verificar a integridade dos resultados alterados entre Silver e Gold. Período: 2020 a 2025.

## 1. Disponibilidade na Bronze

Verifica se agosto de 2022 existe no esquema de `bronze.exames_geral_raw`. A ausência da coluna indica que essa competência não chegou à Silver por essa extração.

In [0]:
from functools import reduce

from pyspark.sql import DataFrame
from pyspark.sql import functions as F

CATALOGO = "`tcc-analise-siscan-10crs`"
ANOS = range(2020, 2026)
MESES_ANO = tuple(range(1, 13))
MESES_CAMPANHA = (9, 10, 11, 12)

TABELAS_SILVER = {
    "silver.exames_geral": MESES_ANO,
    "silver.adequabilidade": MESES_ANO,
    "silver.resultados_categoria_mensal": MESES_ANO,
    "silver.outubro_rosa_faixa_etaria": MESES_CAMPANHA,
}
TABELAS_GOLD = {
    "gold.fato_exames_temporais_territoriais": MESES_ANO,
    "gold.fato_outubro_rosa_producao_municipio_ano": MESES_CAMPANHA,
}


def read_table(name: str) -> DataFrame:
    return spark.table(f"{CATALOGO}.{name}")


def monthly_observations(name: str) -> DataFrame:
    df = read_table(name)
    if name != "gold.fato_outubro_rosa_producao_municipio_ano":
        return df

    months = {
        "exames_setembro": 9, "exames_outubro": 10,
        "exames_novembro": 11, "exames_dezembro": 12,
    }
    month_map = F.create_map(*[F.lit(item) for pair in months.items() for item in pair])
    return (
        df.unpivot(
            ids=["codigo_municipio", "ano"],
            values=list(months),
            variableColumnName="mes_col",
            valueColumnName="quantidade",
        )
        .filter(F.col("quantidade").isNotNull())
        .withColumn("mes", F.element_at(month_map, F.col("mes_col")))
    )


colunas_2022 = sorted(
    name for name in read_table("bronze.exames_geral_raw").columns
    if name.strip().endswith("/2022")
)
print("Competências de 2022:", ", ".join(colunas_2022))
contem_agosto = any(
    name.strip().upper() in {"AGOSTO/2022", "AGO/2022"}
    for name in colunas_2022
)
print(f"Agosto/2022 na Bronze: {'presente' if contem_agosto else 'ausente'}")

## 2. Cobertura temporal da Silver

Compara as competências presentes com o calendário esperado de cada tabela. A ausência de um mês em uma extração por categoria não implica ausência de exames na série geral.

In [0]:
def coverage(name: str, expected_months: tuple[int, ...]) -> DataFrame:
    calendar = spark.createDataFrame([(year,) for year in ANOS], "ano int")
    monthly = (
        monthly_observations(name)
        .groupBy("ano")
        .agg(
            F.sort_array(F.collect_set("mes")).alias("meses_presentes"),
            F.count("*").alias("registros"),
        )
    )
    expected = F.array(*[F.lit(month) for month in expected_months])
    return (
        calendar.join(monthly, "ano", "left")
        .withColumn("meses_presentes", F.coalesce(F.col("meses_presentes"), F.expr("CAST(array() AS ARRAY<INT>)")))
        .select(
            F.lit(name).alias("tabela"),
            "ano",
            expected.alias("meses_esperados"),
            "meses_presentes",
            F.size("meses_presentes").alias("qtd_meses"),
            F.array_except(expected, "meses_presentes").alias("meses_faltantes"),
            F.coalesce(F.col("registros"), F.lit(0)).alias("registros"),
        )
    )


def coverage_report(tables: dict[str, tuple[int, ...]]) -> DataFrame:
    reports = [coverage(name, months) for name, months in tables.items()]
    return reduce(lambda left, right: left.unionByName(right), reports).orderBy("tabela", "ano")


resumo_silver = coverage_report(TABELAS_SILVER)
display(resumo_silver)

## 3. Agosto de 2022

Detalha os meses de 2022 nas tabelas com escopo anual completo. A tabela Outubro Rosa não é incluída, pois agosto está fora do seu recorte.

In [0]:
def august_2022_report(tables: dict[str, tuple[int, ...]]) -> DataFrame:
    full_year_tables = [name for name, months in tables.items() if 8 in months]
    reports = [
        read_table(name)
        .filter(F.col("ano") == 2022)
        .groupBy("mes")
        .agg(F.count("*").alias("registros"))
        .select(F.lit(name).alias("tabela"), "mes", "registros")
        for name in full_year_tables
    ]
    return reduce(lambda left, right: left.unionByName(right), reports).orderBy("tabela", "mes")


display(august_2022_report(TABELAS_SILVER))
display(
    resumo_silver.filter(
        (F.col("ano") == 2022) & F.array_contains("meses_esperados", 8)
    ).select("tabela", F.array_contains("meses_presentes", 8).alias("agosto_presente"))
)

## 4. Cobertura temporal da Gold

A série geral considera os 12 meses do ano. A análise Outubro Rosa inclui setembro a dezembro, conforme a transformação na Gold.

O fato de produção da campanha possui uma coluna por mês; a auditoria converte essas colunas para linhas e considera apenas quantidades não nulas.

In [0]:
resumo_gold = coverage_report(TABELAS_GOLD)
display(resumo_gold)
display(
    resumo_gold.filter(
        (F.col("ano") == 2022) & F.array_contains("meses_esperados", 8)
    ).select("tabela", F.array_contains("meses_presentes", 8).alias("agosto_presente"))
)

## 5. Integridade dos exames alterados

Compara o numerador da Gold com a quantidade da Silver por código do município e ano. O full outer join identifica diferenças de valores e chaves presentes em apenas uma camada.

In [0]:
keys = ["codigo_municipio", "ano"]

gold = (
    read_table("gold.fato_exames_alterados")
    .groupBy(*keys)
    .agg(F.sum("quantidade_alterados").alias("alterados_gold"))
)
silver = (
    read_table("silver.resultados_citopatologicos")
    .groupBy(*keys)
    .agg(F.sum("quantidade").alias("alterados_silver"))
)

divergencias = (
    gold.join(silver, keys, "full_outer")
    .withColumn("diff", F.col("alterados_gold") - F.col("alterados_silver"))
    .filter(
        F.col("alterados_gold").isNull()
        | F.col("alterados_silver").isNull()
        | (F.col("diff") != 0)
    )
)
n_divergencias = divergencias.count()
if n_divergencias:
    display(divergencias.orderBy(*keys))
    raise ValueError(f"{n_divergencias} divergências entre Gold e Silver.")

print("Numerador de exames alterados: Gold e Silver conferem.")